In [5]:
import sys
sys.path.append('..')
from search.retrieval import *

corpus, inverted_index, bm25_data = load_indices()
faiss_index, faiss_doc_ids, model = load_faiss()
graph, synonyms = load_graph()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/katharinazeilnhofer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loaded: 18114 documents, 76513 tokens


No sentence-transformers model found with name T-Systems-onsite/cross-en-de-roberta-sentence-transformer. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: T-Systems-onsite/cross-en-de-roberta-sentence-transformer
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS loaded: 18114 vectors
Graph loaded: 38694 nodes, 82647 edges


In [6]:
query = "Kartoffelsuppe"

print("=== BM25 ===")
for doc_id, title, score in bm25_search(query, corpus, bm25_data):
    print(f"  [{score:.2f}] {title}")

print("\n=== Semantic Search (FAISS) ===")
for doc_id, title, dist in faiss_search(query, corpus, faiss_index, faiss_doc_ids, model):
    print(f"  [{dist:.4f}] {title}")

print("\n=== Graph Search ===")
for doc_id, title in graph_search(["Kartoffel", "Zwiebel", "Speck"], graph, corpus):
    print(f"  {title}")

=== BM25 ===
  [10.55] Kartoffelsuppe (vegan)
  [10.55] Vegetarische Kartoffelsuppe
  [10.48] Diät Kartoffelsuppe
  [10.48] Kartoffelsuppe (Diät)
  [10.48] Kartoffelsuppe fein

=== Semantic Search (FAISS) ===
  [9.2021] Vegetarische Kartoffelsuppe
  [9.2021] Kartoffelsuppe (vegan)
  [19.6648] Vegane Kartoffelsuppe
  [25.9733] Kartoffelsuppen
  [26.5040] Kartoffelsuppe fein

=== Graph Search ===
  Kartoffelsuppe mit Rookworst
  Kartoffel-Zwiebel-Eintopf mit Nelkenkäse
  Buttermilch-Kartoffeln
  Erbsensuppe mit geräucherten Rippen
  Stamppot mit rohen Endivien


In [7]:
# without graph filter 
print("=== Hybrid: 'Suppe' ===")
for _, title, score in hybrid_search("Suppe", corpus, bm25_data, faiss_index, faiss_doc_ids, model):
    print(f"  [{score:.4f}] {title}")

# with graph filter
print("\n=== Hybrid: 'Suppe' + Filter 'Kartoffel' ===")
for _, title, score in hybrid_search("Suppe", corpus, bm25_data, faiss_index, faiss_doc_ids, model, 
                                      graph=graph, filter_zutaten=["Kartoffel"]):
    print(f"  [{score:.4f}] {title}")

=== Hybrid: 'Suppe' ===
  [0.8837] Sangria-Suppe
  [0.8233] Sburrita di merluzzo
  [0.7808] Rote Bete Suppe mit Meerrettich
  [0.7795] Klare Suppe mit Ei
  [0.7738] Brunnenkressesuppe mit Käse

=== Hybrid: 'Suppe' + Filter 'Kartoffel' ===
  [0.7701] Caldo verde
  [0.7572] Kartoffelcremesuppe
  [0.7453] Fischsuppe mit Gemüse
  [0.7433] Kräutersuppe mit Ei
  [0.7246] Klare, leichte Kartoffelsuppe
